In [4]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import time
import tracemalloc
import torch.optim as optim

import torch.nn as nn

import torch.nn.functional as F

def cosine_similarity_loss(pred, target):
    """
    Compute the cosine similarity loss between predictions and targets.

    Args:
        pred: Predicted values (shape: [batch_size, ...]).
        target: True values (shape: [batch_size, ...]).

    Returns:
        loss: Cosine similarity loss.
    """
    # Flatten the tensors to compute cosine similarity
    pred_flat = pred.view(pred.size(0), -1)
    target_flat = target.view(target.size(0), -1)

    # Compute cosine similarity
    cosine_sim = F.cosine_similarity(pred_flat, target_flat, dim=1)

    # Transform to loss (minimize the angle between vectors)
    loss = 1 - cosine_sim.mean()
    return loss
    
class SequentialPredictionModel(nn.Module):
    def __init__(self, covariate_dim, treatment_dim, hidden_dim, output_dim, dropout_prob=0.5):
        super(SequentialPredictionModel, self).__init__()
        # Network to predict X2_hat from X1 and A1
        self.net1 = nn.Sequential(
            nn.Linear(covariate_dim + treatment_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, covariate_dim)
        )
        # Network to predict X3_hat from X2_hat, A2, and X1
        self.net2 = nn.Sequential(
            nn.Linear(covariate_dim + treatment_dim + covariate_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, covariate_dim)
        )
        # Network to predict Y from X3_hat, A3, X1, A1, and A2
        self.net3 = nn.Sequential(
            nn.Linear(covariate_dim + treatment_dim + covariate_dim + 2 * treatment_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, X1, A):
        A1 = A[:, 0].unsqueeze(1)
        A2 = A[:, 1].unsqueeze(1)
        A3 = A[:, 2].unsqueeze(1)

        # Predict X2_hat
        X2_hat = self.net1(torch.cat([X1, A1], dim=1))

        # Predict X3_hat
        X3_hat = self.net2(torch.cat([X2_hat, A2, X1], dim=1))

        # Predict Y
        y_pred = self.net3(torch.cat([X3_hat, A3, X1, A1, A2], dim=1))
        return X2_hat, X3_hat, y_pred

In [ ]:
def process_single_file(file_path):
    """Process a single data file and return causal effects"""
    # Load and preprocess data
    data = pd.read_csv(file_path)
    y = data["Y_glomerular_filtration"].values
    treatments = data[["treatment_1", "treatment_2", "treatment_3"]].values
    propensity_scores = data[["ps_treatment_1", "ps_treatment_2", "ps_treatment_3"]].values
    covariates = data.drop(columns=["Y_glomerular_filtration", "treatment_1", "treatment_2", "treatment_3", 
                                  "ps_treatment_1", "ps_treatment_2", "ps_treatment_3"]).values
    covariates = covariates.reshape(-1, 3, 4)
    
    # Split data
    X_train, X_val, y_train, y_val, A_train, A_val, ps_train, ps_val = train_test_split(
        covariates, y, treatments, propensity_scores, test_size=0.2, random_state=42)
    
    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    A_train = torch.tensor(A_train, dtype=torch.float32)
    A_val = torch.tensor(A_val, dtype=torch.float32)
    ps_train = torch.tensor(ps_train, dtype=torch.float32)
    ps_val = torch.tensor(ps_val, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    y_val = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
    
    # Compute weights
    def compute_weights(A, ps):
        weights = []
        for i in range(3):
            w = A[:, i] / ps[:, i] + (1 - A[:, i]) / (1 - ps[:, i])
            weights.append(w)
        return torch.stack(weights, dim=1).prod(dim=1)
    
    combined_weight_train = compute_weights(A_train, ps_train)
    combined_weight_val = compute_weights(A_val, ps_val)
    
    # Create DataLoaders
    train_dataset = TensorDataset(X_train, A_train, combined_weight_train, y_train)
    val_dataset = TensorDataset(X_val, A_val, combined_weight_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # Initialize and train model
    model = SequentialPredictionModel(
        covariate_dim=4,
        treatment_dim=1,
        hidden_dim=128,
        output_dim=1,
        dropout_prob=0.5
    )
    
    # Train model (using your existing training loop)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    tracemalloc.start()
    start_time = time.time()
    
    for epoch in range(20):
        model.train()
        for X_batch, A_batch, weights_batch, y_batch in train_loader:
            optimizer.zero_grad()
            X1 = X_batch[:, 0, :]
            X2_hat, X3_hat, y_pred = model(X1, A_batch)
            
            loss_X2 = criterion(X2_hat, X_batch[:, 1, :])
            loss_X3 = criterion(X3_hat, X_batch[:, 2, :])
            loss_y = criterion(y_pred, y_batch)
            total_loss = torch.mean(weights_batch**2 * (loss_X2 + loss_X3 + loss_y))
            
            total_loss.backward()
            optimizer.step()

    
    peak_memory = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    training_time = time.time() - start_time
    
    # Calculate causal effects
    def estimate_effect(model, X_val, A_val, treatment_pattern):
        model.eval()
        device = next(model.parameters()).device
        A_treatment = torch.tensor(treatment_pattern, dtype=torch.float32).view(1, 3).to(device)
        A_control = torch.zeros(1, 3).to(device)
        
        y_treatment, y_control = [], []
        with torch.no_grad():
            for X_sample in X_val:
                X1 = X_sample[0].unsqueeze(0).to(device)
                _, _, y_t = model(X1, A_treatment)
                _, _, y_c = model(X1, A_control)
                y_treatment.append(y_t.item())
                y_control.append(y_c.item())
        
        return np.mean(np.array(y_treatment) - np.array(y_control))
    
    effect_111 = estimate_effect(model, X_val, A_val, [1,1,1])
    effect_011 = estimate_effect(model, X_val, A_val, [0,1,1])
    
    return {
        'file_name': os.path.basename(file_path),
        'effect_111': effect_111,
        'effect_011': effect_011,
        'training_time': training_time,
        'peak_memory': peak_memory / 1024**2
    }

def process_directory(directory_path, output_csv):
    """Process all CSV files in directory and save results"""
    results = []
    processed_files = set()
    
    # Check if output file exists to resume processing
    if os.path.exists(output_csv):
        existing_results = pd.read_csv(output_csv)
        processed_files = set(existing_results['file_name'].tolist())
        results = existing_results.to_dict('records')
    
    # Get all CSV files in directory
    files = sorted([f for f in os.listdir(directory_path) if f.endswith('.csv')])
    
    for i, filename in enumerate(files):
        if filename in processed_files:
            print(f"Skipping already processed file: {filename}")
            continue
            
        file_path = os.path.join(directory_path, filename)
        print(f"\nProcessing file {i+1}/{len(files)}: {filename}")
        
        try:
            result = process_single_file(file_path)
            results.append(result)
            
            # Save after each file in case of interruption
            pd.DataFrame(results).to_csv(output_csv, index=False)
            print(f"Saved results for {filename}")
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            continue
    
    return pd.DataFrame(results)

# Main execution
if __name__ == "__main__":
    data_directory = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/parallel_linear_sd_Z_07_sd_eps_3"
    output_file = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/sequential_model_parallel_linear_sd_Z_07_sd_eps_3_results.csv"
    
    print(f"Starting processing of directory: {data_directory}")
    results_df = process_directory(data_directory, output_file)
    print(f"\nProcessing complete. Results saved to {output_file}")
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(f"Processed {len(results_df)} files")
    if len(results_df) > 0:
        print(f"Average Effect [1,1,1]: {results_df['effect_111'].mean():.4f}")
        print(f"Average Effect [0,1,1]: {results_df['effect_011'].mean():.4f}")
        print(f"Average Training Time: {results_df['training_time'].mean():.2f} seconds")
        print(f"Average Peak Memory: {results_df['peak_memory'].mean():.2f} MiB")

Starting processing of directory: /gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/parallel_linear_sd_Z_07_sd_eps_3

Processing file 1/1000: simulated_data_2_seed_20250001_sdZ_07_sdEps_3.csv
Saved results for simulated_data_2_seed_20250001_sdZ_07_sdEps_3.csv

Processing file 2/1000: simulated_data_2_seed_20250002_sdZ_07_sdEps_3.csv
Saved results for simulated_data_2_seed_20250002_sdZ_07_sdEps_3.csv

Processing file 3/1000: simulated_data_2_seed_20250003_sdZ_07_sdEps_3.csv
Saved results for simulated_data_2_seed_20250003_sdZ_07_sdEps_3.csv

Processing file 4/1000: simulated_data_2_seed_20250004_sdZ_07_sdEps_3.csv
Saved results for simulated_data_2_seed_20250004_sdZ_07_sdEps_3.csv

Processing file 5/1000: simulated_data_2_seed_20250005_sdZ_07_sdEps_3.csv
Saved results for simulated_data_2_seed_20250005_sdZ_07_sdEps_3.csv

Processing file 6/1000: simulated_data_2_seed_20250006_sdZ_07_sdEps_3.csv
Saved results for simulated_data_2_seed_20250006_sdZ_07_sdEps_3

In [ ]:
# Main execution
if __name__ == "__main__":
    data_directory = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/parallel_nonlinear_sd_Z_01_sd_eps_1"
    output_file = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/sequential_model_parallel_nonlinear_sd_Z_01_sd_eps_1_results.csv"
    
    print(f"Starting processing of directory: {data_directory}")
    results_df = process_directory(data_directory, output_file)
    print(f"\nProcessing complete. Results saved to {output_file}")
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(f"Processed {len(results_df)} files")
    if len(results_df) > 0:
        print(f"Average Effect [1,1,1]: {results_df['effect_111'].mean():.4f}")
        print(f"Average Effect [0,1,1]: {results_df['effect_011'].mean():.4f}")
        print(f"Average Training Time: {results_df['training_time'].mean():.2f} seconds")
        print(f"Average Peak Memory: {results_df['peak_memory'].mean():.2f} MiB")

In [ ]:
# Main execution
if __name__ == "__main__":
    data_directory = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/parallel_linear_sd_Z_01_sd_eps_3"
    output_file = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/sequential_model_parallel_linear_sd_Z_01_sd_eps_3_results.csv"
    
    print(f"Starting processing of directory: {data_directory}")
    results_df = process_directory(data_directory, output_file)
    print(f"\nProcessing complete. Results saved to {output_file}")
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(f"Processed {len(results_df)} files")
    if len(results_df) > 0:
        print(f"Average Effect [1,1,1]: {results_df['effect_111'].mean():.4f}")
        print(f"Average Effect [0,1,1]: {results_df['effect_011'].mean():.4f}")
        print(f"Average Training Time: {results_df['training_time'].mean():.2f} seconds")
        print(f"Average Peak Memory: {results_df['peak_memory'].mean():.2f} MiB")

In [5]:
import pandas as pd
import numpy as np

def calculate_metrics(csv_path, true_effect_111, true_effect_011):
    """
    Calculate performance metrics for causal effect estimates
    
    Args:
        csv_path: Path to results CSV file
        true_effect_111: True causal effect for [1,1,1] vs [0,0,0]
        true_effect_011: True causal effect for [0,1,1] vs [0,0,0]
    
    Returns:
        Dictionary containing all six metrics
    """
    # Read results
    df = pd.read_csv(csv_path)
    
    # Calculate metrics for effect_111
    mean_hat_111 = df['effect_111'].mean()
    rel_bias_111 = (mean_hat_111 - true_effect_111) / true_effect_111
    mcsd_111 = df['effect_111'].std(ddof=1)  # Using H-1 in denominator
    rmse_111 = np.sqrt(((df['effect_111'] - true_effect_111)**2).mean())
    
    # Calculate metrics for effect_011
    mean_hat_011 = df['effect_011'].mean()
    rel_bias_011 = (mean_hat_011 - true_effect_011) / true_effect_011
    mcsd_011 = df['effect_011'].std(ddof=1)  # Using H-1 in denominator
    rmse_011 = np.sqrt(((df['effect_011'] - true_effect_011)**2).mean())
    
    return {
        'effect_111': {
            'relative_bias': rel_bias_111,
            'mcsd': mcsd_111,
            'rmse': rmse_111
        },
        'effect_011': {
            'relative_bias': rel_bias_011,
            'mcsd': mcsd_011,
            'rmse': rmse_011
        }
    }

# Example usage
if __name__ == "__main__":
    # Replace with your actual CSV path and true effect values
    results_csv = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/sequential_model_parallel_linear_sd_Z_07_sd_eps_3_results.csv"
    
    # You'll need to provide these true effect values from your large simulation
    true_effect_111 = 5.0  # Replace with your actual value
    true_effect_011 = 3.0  # Replace with your actual value
    
    metrics = calculate_metrics(results_csv, 5.041, 3.021)
    
    # Print results
    print("Performance Metrics:")
    print("\nFor Effect [1,1,1] vs [0,0,0]:")
    print(f"Relative Bias: {metrics['effect_111']['relative_bias']:.4f}")
    print(f"Monte Carlo SD: {metrics['effect_111']['mcsd']:.4f}")
    print(f"RMSE: {metrics['effect_111']['rmse']:.4f}")
    
    print("\nFor Effect [0,1,1] vs [0,0,0]:")
    print(f"Relative Bias: {metrics['effect_011']['relative_bias']:.4f}")
    print(f"Monte Carlo SD: {metrics['effect_011']['mcsd']:.4f}")
    print(f"RMSE: {metrics['effect_011']['rmse']:.4f}")

Performance Metrics:

For Effect [1,1,1] vs [0,0,0]:
Relative Bias: 1.4938
Monte Carlo SD: 1.2025
RMSE: 7.6257

For Effect [0,1,1] vs [0,0,0]:
Relative Bias: 2.2332
Monte Carlo SD: 0.9443
RMSE: 6.8123
